# Train text classification model on DBpedia dataset


## 1) Thiết lập môi trường (local hoặc Colab)

Notebook tự động kiểm tra Colab hay local, chuyển vào repository và cấu hình sys.path.


In [ ]:
import os, sys

# Nếu đang trong tình trạng current dir hỏng hap khong
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = None

print("Current dir:", cwd)

if cwd is None or not os.path.exists(cwd):
    # chuyển về thu muc cho an toan
    os.chdir('/content')
    print("Changed to /content")

print("cwd now", os.getcwd())

# then continue clone logic
is_colab = os.path.exists('/content')
if is_colab:
    target = '/content/text-classification'
    if os.path.exists(target):
        !rm -rf /content/text-classification
    !git clone https://github.com/huynhunguyen/text-classification.git /content/text-classification
    %cd /content/text-classification
    print('Running in Colab mode. Current dir:', os.getcwd())
else:
    # local mode
    print('Running in local mode. Current dir:', os.getcwd())
    if not os.path.exists(os.path.join(os.getcwd(), 'src')):
        raise RuntimeError('Chưa ở đúng thư mục repo; hãy chuyển tới db_text_class')

repo_root = os.getcwd()
sys.path.insert(0, os.path.join(repo_root, 'src'))
print('Repo root:', repo_root)
print('sys.path[0] =', sys.path[0])


Current dir: /content
cwd now /content
Cloning into '/content/text-classification'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 36 (delta 0), reused 4 (delta 0), pack-reused 30 (from 1)
Receiving objects: 100% (36/36), 88.05 MiB | 40.18 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/text-classification
Running in Colab mode. Current dir: /content/text-classification
Repo root: /content/text-classification
sys.path[0] = /content/text-classification/src


## 2) Cài đặt dependencies

Chạy một lần để cài thư viện cần thiết.


In [ ]:
import os

# Gỡ cài đặt sạch sẽ
!pip uninstall -y torch torchvision torchaudio torchtext fastai timm || true

# Cài đặt bộ phiên bản khớp nhau hoàn toàn cho CUDA 12.1
!pip install -q torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torchtext==0.18.0
!pip install -q "numpy>=1.26.0,<2.1"
!pip install -q datasets



Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: fastai 2.8.7
Uninstalling fastai-2.8.7:
  Successfully uninstalled fastai-2.8.7
Found existing installation: timm 1.0.25
Uninstalling timm-1.0.25:
  Successfully uninstalled timm-1.0.25
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 74.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 123.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/

## 3) Mount Google Drive (nếu dùng Colab)


In [ ]:
import os

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('Drive mounted: /content/drive')
else:
    print('Không phải Colab; không cần mount Drive')


Mounted at /content/drive
Drive mounted: /content/drive


## 4) Chuẩn bị dữ liệu DBpedia


In [ ]:
from datasets import load_dataset
import os

# Tạo thư mục data nếu chưa có
os.makedirs('data', exist_ok=True)

print("Đang tải dataset từ Hugging Face...")
dataset = load_dataset("dbpedia_14")

# Chuyển đổi và lưu thành CSV
print("Đang lưu dataset thành tệp CSV...")
dataset['train'].to_csv('data/train.csv', index=False)
dataset['test'].to_csv('data/test.csv', index=False)

print("Hoàn tất! Các tệp dữ liệu đã sẵn sàng trong thư mục data/")
!ls -lh data

Đang tải dataset từ Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Đang lưu dataset thành tệp CSV...


Creating CSV from Arrow format:   0%|          | 0/560 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/70 [00:00<?, ?ba/s]

Hoàn tất! Các tệp dữ liệu đã sẵn sàng trong thư mục data/
total 185M
-rw-r--r-- 1 root root  146 Mar 20 15:25 classes.txt
-rw-r--r-- 1 root root 1.8K Mar 20 15:25 readme.txt
-rw-r--r-- 1 root root  21M Mar 20 15:31 test.csv
-rw-r--r-- 1 root root 164M Mar 20 15:31 train.csv


## 5) Chạy train script

Điều chỉnh tham số theo nhu cầu, bạn có thể sửa trong src/train.py nếu script không có tham số.


In [ ]:
%cd /content/text-classification
!python src/train.py

/content/text-classification
/usr/local/lib/python3.12/dist-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.12/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.12/dist-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS

## 6) Kiểm tra model và lưu model ra Drive


In [ ]:
!find models -maxdepth 3 -type f | head -n 20
!mkdir -p /content/drive/MyDrive/text_classification/models
!cp -r models/* /content/drive/MyDrive/text_classification/models


models/rnn/001.pth
models/transformer/004.pth
models/transformer/005.pth
models/transformer/003.pth
models/transformer/001.pth
models/transformer/002.pth


## 7) Dự đoán bằng script và đánh giá

Sử dụng `src/predict.py` nếu project có hỗ trợ.


In [ ]:
python src/predict.py --model_path models/transformer/005.pth --input_path data/test.csv --output_path predictions.csv
head -n 20 predictions.csv


SyntaxError: invalid decimal literal (3862643450.py, line 1)